# TOAP KV-Bridge Benchmark (Colab)

Real KV-bridge experiment: **recompute** (re-prefill prefix+query) vs **KV-bridge** (reuse the
transferred prefix KV and prefill only the query). Per shared-context length we measure:

1. **prefill latency** — recompute vs bridge (the compute win),
2. **transfer cost** — KV serialize/deserialize time (reported separately, never hidden),
3. **KV byte size vs text size** — the honest storage/bandwidth cost,
4. **correctness** — greedy tokens must match the recompute baseline exactly (losslessness).

Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU), then **Run all**. A model matrix
spanning MHA (GPT-2), Pythia, GQA (Qwen2.5), and a 4-bit 7B runs with per-model error isolation —
if one model fails or OOMs, the rest still complete. At the end, `kv_bench_all.json` is produced;
send it back to fold into the TOAP paper.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Set Runtime -> Change runtime type -> T4 GPU, then Run all.'

In [ ]:
# Dependencies. bitsandbytes + accelerate enable the optional 4-bit 7B model.
!pip -q install -U transformers accelerate bitsandbytes
!git clone https://github.com/parnish007/TOAP.git
%cd TOAP/benchmark/kv_bridge
import transformers, torch
print('transformers', transformers.__version__, '| torch', torch.__version__)

In [ ]:
# Smoke test on the smallest model first — confirms the pipeline works before the big runs.
# Asserts non-empty rows AND token-lossless match, so a version/API bug is caught in seconds
# (not after the full matrix). If this fails, STOP and report the printed error.
import json, subprocess
subprocess.run(['python', 'kv_bench.py', '--model', 'sshleifer/tiny-gpt2',
                '--prefix-tokens', '64', '128', '--new-tokens', '8', '--repeats', '3',
                '--out', 'smoke.json'], check=True)
_sm = json.load(open('smoke.json'))
assert _sm['rows'], 'SMOKE FAILED: no rows produced (a cache/API bug — do not run the matrix yet).'
assert all(r['outputs_match'] for r in _sm['rows']), 'SMOKE FAILED: KV-bridge not token-lossless.'
print('smoke OK:', [(r['prefix_tokens'], round(r['prefill_speedup'], 2), r['outputs_match'])
                    for r in _sm['rows']])

In [ ]:
# Model matrix. Each entry runs in isolation; a failure (OOM, download, arch quirk) is caught and
# the matrix continues. Lengths default to auto (powers-of-two within each model's context window).
import json, subprocess, os, glob, time

MATRIX = [
    # name, args
    ('gpt2',                          ['--model', 'gpt2']),                       # MHA, ctx 1024
    ('gpt2-large',                    ['--model', 'gpt2-large']),                 # bigger MHA
    ('EleutherAI/pythia-410m',        ['--model', 'EleutherAI/pythia-410m']),     # NeoX, ctx 2048
    ('EleutherAI/pythia-1.4b',        ['--model', 'EleutherAI/pythia-1.4b']),     # bigger NeoX
    ('Qwen/Qwen2.5-0.5B-Instruct',    ['--model', 'Qwen/Qwen2.5-0.5B-Instruct']), # GQA, long ctx
    ('Qwen/Qwen2.5-1.5B-Instruct',    ['--model', 'Qwen/Qwen2.5-1.5B-Instruct']), # GQA
    ('mistralai/Mistral-7B-Instruct-v0.3', ['--model', 'mistralai/Mistral-7B-Instruct-v0.3', '--load-in-4bit']),  # 7B GQA, 4-bit
]
COMMON = ['--new-tokens', '32', '--repeats', '5']

results = []
for name, args in MATRIX:
    safe = name.replace('/', '__')
    out = f'res_{safe}.json'
    print(f'\n===== {name} =====', flush=True)
    t0 = time.time()
    try:
        subprocess.run(['python', 'kv_bench.py', *args, *COMMON, '--out', out],
                       check=True, timeout=1800)
        if os.path.exists(out):
            results.append(json.load(open(out)))
            print(f'[ok] {name} in {time.time()-t0:.0f}s')
    except subprocess.TimeoutExpired:
        print(f'[skip] {name}: timed out')
    except subprocess.CalledProcessError as e:
        print(f'[skip] {name}: exited {e.returncode}')
    except Exception as e:
        print(f'[skip] {name}: {type(e).__name__}: {e}')
    # free VRAM between models
    torch.cuda.empty_cache()

with open('kv_bench_all.json', 'w') as f:
    json.dump({'runs': results}, f, indent=2)
print(f'\n[written] kv_bench_all.json with {len(results)} model runs')

In [ ]:
# Quick summary table across all models that completed.
import json
data = json.load(open('kv_bench_all.json'))
print(f"{'model':36} {'dtype':10} {'prefix':>7} {'speedup':>8} {'kv/text':>8} {'match':>6}")
print('-'*82)
for run in data['runs']:
    for r in run['rows']:
        print(f"{run['model'][:36]:36} {run['dtype'][:10]:10} {r['prefix_tokens']:>7} "
              f"{r['prefill_speedup']:>7.2f}x {(r['kv_vs_text_ratio'] or 0):>6.0f}x "
              f"{str(r['outputs_match']):>6}")

In [ ]:
from google.colab import files
files.download('kv_bench_all.json')